# Map and save mapped data

Companion to `save_unmapped_data.ipynb`. Reads the already-saved unmapped chunks from

```
payment_processing_research_data/<bureau>/<split>/unmapped/part-NNNNN.parquet
```

runs the FE2 `MapperV2` for the bureau, and writes the result back to

```
payment_processing_research_data/<bureau>/<split>/mapped/part-NNNNN.parquet
```

Loop order matches `save_unmapped_data.ipynb`: **train -> valid -> test**, and within each split, **equifax -> experian -> transunion**. So train data for all three bureaus gets mapped before any valid data.

No Snowflake access needed -- this is purely a local read -> map -> write pipeline.

In [2]:
%pip install model-engine

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: http://zamlpkgs-nginx/artifactory/api/pypi/zest_pypi/simple/
  Using cached model_engine-2.1.4-py3-none-any.whl
  Using cached feature_engine_parts-2.0.2-py3-none-any.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [model-engine] [model-engine]

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import gc
import warnings
from pathlib import Path

import pandas as pd

from configs import EQUIFAX, EXPERIAN, TRANSUNION, DATA_DIR, unmapped_dir, mapped_dir
from helpers import build_mappers, save_in_chunks, CHUNK_SIZE

warnings.filterwarnings('ignore')

print('DATA_DIR   =', DATA_DIR)
print('CHUNK_SIZE =', f'{CHUNK_SIZE:,}', 'rows per parquet file')

/home/jag/.local/lib/python3.10/site-packages/snowflake/connector/options.py:128: UserWarning: You have an incompatible version of 'pyarrow' installed (11.0.0), please install a version that adheres to: 'pyarrow>=14.0.1; extra == "pandas"'
  warn_incompatible_dep(


DATA_DIR   = /home/jag/payment-processor-research/payment_processing_research_data
CHUNK_SIZE = 100,000 rows per parquet file


In [2]:
# One MapperV2 per bureau, built via model_engine.load_asset (in helpers.py).
mappers = build_mappers()

equifax: mapper built from equifax/cms_6/fe2/trade.json
experian: mapper built from experian/arf7/fe2/trade.json
transunion: mapper built from transunion/TU4R/fe2/trade.json


In [3]:
# save_in_chunks + CHUNK_SIZE come from helpers (imported above).
# Nothing to define here; this cell is kept as a placeholder for clarity.

In [ ]:
# OUTER loop: splits in order train -> valid -> test.
# INNER loop: equifax -> experian -> transunion.
#
# STREAMING: for each (bureau, split) we walk the unmapped parquet files
# one at a time -- read one chunk -> map it -> write the mapped chunk.
# Memory stays bounded to ~one chunk (CHUNK_SIZE rows) instead of loading
# the entire 40-60M-row unmapped dataset at once. Each input
# `part-NNNNN.parquet` maps 1:1 to an output `part-NNNNN.parquet`.
SPLITS  = ['train', 'valid', 'test']
BUREAUS = [EQUIFAX, EXPERIAN, TRANSUNION]

for split in SPLITS:
    print(f'\n##### SPLIT = {split} #####')
    for cfg in BUREAUS:
        bureau  = cfg['bureau']
        in_dir  = Path(unmapped_dir(cfg, split))
        out_dir = Path(mapped_dir(cfg, split))

        print(f'\n=== {bureau}/{split} ===')
        parts = sorted(in_dir.glob('part-*.parquet')) if in_dir.exists() else []
        if not parts:
            print(f'[{bureau}/{split}]   MISSING unmapped chunks at {in_dir} -- run save_unmapped_data.ipynb first')
            continue

        # Clear any previous run's mapped chunks so reruns are idempotent
        # (matches save_in_chunks's behavior).
        out_dir.mkdir(parents=True, exist_ok=True)
        for old in out_dir.glob('part-*.parquet'):
            old.unlink()

        total_rows = 0
        for i, in_path in enumerate(parts):
            # 1. Read one unmapped chunk
            chunk = pd.read_parquet(in_path)

            # 2. Map this chunk only
            mapped = mappers[bureau].transform(chunk)

            # 3. Save to the matching output part file (same NNNNN index)
            out_path = out_dir / f'part-{i:05d}.parquet'
            mapped.to_parquet(out_path, index=False)
            total_rows += len(mapped)

            # Periodic progress + final tick so long runs stay informative
            if (i + 1) % 50 == 0 or i == len(parts) - 1:
                print(f'[{bureau}/{split}]   {i + 1}/{len(parts)} chunks mapped '
                      f'({total_rows:,} rows so far)')

            # 4. Free memory before the next chunk
            del chunk, mapped
            gc.collect()

        print(f'[{bureau}/{split}]   done: {len(parts)} chunks, {total_rows:,} rows -> {out_dir}')


##### SPLIT = train #####

=== equifax/train ===
[equifax/train]   50/582 chunks mapped (5,000,000 rows so far)
[equifax/train]   100/582 chunks mapped (10,000,000 rows so far)
[equifax/train]   150/582 chunks mapped (15,000,000 rows so far)
[equifax/train]   200/582 chunks mapped (20,000,000 rows so far)
[equifax/train]   250/582 chunks mapped (25,000,000 rows so far)
[equifax/train]   300/582 chunks mapped (30,000,000 rows so far)
[equifax/train]   350/582 chunks mapped (35,000,000 rows so far)
[equifax/train]   400/582 chunks mapped (40,000,000 rows so far)
[equifax/train]   450/582 chunks mapped (45,000,000 rows so far)
[equifax/train]   500/582 chunks mapped (50,000,000 rows so far)
[equifax/train]   550/582 chunks mapped (55,000,000 rows so far)
[equifax/train]   582/582 chunks mapped (58,130,132 rows so far)
[equifax/train]   done: 582 chunks, 58,130,132 rows -> /home/jag/payment-processor-research/payment_processing_research_data/equifax/train/mapped

=== experian/train ===
[e

In [ ]:
# Verify the chunked output -- one row per (bureau, split) showing chunks + rows + size.
import pyarrow.dataset as ds

rows = []
for split in SPLITS:
    for cfg in BUREAUS:
        d = Path(mapped_dir(cfg, split))
        parts = sorted(d.glob('part-*.parquet')) if d.exists() else []
        if not parts:
            rows.append({'bureau': cfg['bureau'], 'split': split, 'chunks': 0,
                         'rows': 0, 'size_mb': 0.0, 'dir': str(d)})
            continue
        n_rows = ds.dataset(str(d), format='parquet').count_rows()
        total_mb = sum(p.stat().st_size for p in parts) / (1024 * 1024)
        rows.append({'bureau': cfg['bureau'], 'split': split,
                     'chunks': len(parts), 'rows': n_rows,
                     'size_mb': round(total_mb, 1), 'dir': str(d)})

pd.DataFrame(rows)